# Seu primeiro sistema multiagente com CrewAI

Este notebook recria e moderniza o exemplo da aula: uma equipe de três agentes que planeja, escreve e edita um artigo.

- **Planejador de conteúdo:** organiza público, tese, tópicos e informações necessárias.
- **Redator:** transforma o plano em um artigo Markdown.
- **Editor:** revisa clareza, coerência, precisão e formato.

Os conceitos centrais são os mesmos apresentados na aula: `Agent`, `Task`, `Crew`, simulação de papéis, objetivos claros, resultados esperados e execução sequencial.

## Como a colaboração funciona

```text
tema + contexto → planejamento → redação → edição → artigo final
```

Cada agente faz uma tarefa granular. As saídas anteriores são fornecidas como contexto para as tarefas seguintes. A versão atual do CrewAI usa `verbose=True`; o valor numérico `verbose=2` visto em materiais antigos não é necessário.

## 1. Instalar o CrewAI

Execute esta célula uma vez. Se o ambiente solicitar, reinicie o kernel depois da instalação.

In [ ]:
%pip install -q "crewai[litellm]==1.15.12"

## 2. Configurar o modelo

Defina `GROQ_API_KEY` no arquivo `.env` ou no ambiente. O modelo é configurado com `GROQ_MODEL`. Se a chave não estiver no ambiente, o notebook a solicita de forma oculta. Nunca escreva a chave diretamente em uma célula.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("CREWAI_STORAGE_DIR", str(Path.cwd() / ".crewai-data"))

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
if not MODEL.startswith("groq/"):
    MODEL = f"groq/{MODEL}"
print(f"Modelo configurado: {MODEL}")

## 3. Criar os agentes

Assim como na aula, cada agente recebe três elementos principais:

- `role`: sua função na equipe;
- `goal`: o resultado que deve perseguir;
- `backstory`: o contexto profissional que orienta seu comportamento.

A variável `{topico}` será interpolada quando a equipe for iniciada.

In [ ]:
from crewai import Agent, LLM
from crewai.hooks.llm_hooks import register_before_llm_call_hook

def remover_cache_breakpoints_incompativeis(context):
    if MODEL.startswith("groq/"):
        for mensagem in context.messages:
            mensagem.pop("cache_breakpoint", None)
    return None

register_before_llm_call_hook(remover_cache_breakpoints_incompativeis)

MODELO_LLM = LLM(model=MODEL, max_tokens=1200, temperature=0.2)

planejador = Agent(
    role="Planejador de conteúdo sobre {topico}",
    goal=(
        "Criar um plano envolvente, lógico e factualmente responsável para um artigo "
        "sobre {topico}."
    ),
    backstory=(
        "Você é um estrategista editorial experiente. Entende o público, identifica a tese "
        "central e organiza informações complexas em uma narrativa fácil de acompanhar. "
        "Você diferencia fatos fornecidos de pontos que ainda precisam de verificação."
    ),
    llm=MODELO_LLM,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

redator = Agent(
    role="Redator de conteúdo sobre {topico}",
    goal=(
        "Escrever um artigo claro, útil e envolvente sobre {topico}, seguindo o plano "
        "editorial e sem inventar fatos ou fontes."
    ),
    backstory=(
        "Você é um redator técnico que transforma planos e referências em textos naturais. "
        "Explica ideias com exemplos, mantém consistência de tom e sinaliza qualquer afirmação "
        "que precise ser confirmada."
    ),
    llm=MODELO_LLM,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

editor = Agent(
    role="Editor-chefe de conteúdo sobre {topico}",
    goal=(
        "Entregar um artigo final preciso, coerente, bem formatado e adequado ao público, "
        "preservando ressalvas importantes."
    ),
    backstory=(
        "Você é o último responsável antes da publicação. Corrige estrutura, linguagem, "
        "repetição e tom; remove afirmações não sustentadas e não cria referências inexistentes."
    ),
    llm=MODELO_LLM,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

print("Agentes criados: planejador, redator e editor.")

## 4. Criar as tarefas

Cada `Task` define uma descrição, um resultado esperado e o agente responsável. O `expected_output` funciona como um contrato em linguagem natural: quanto mais concreto, mais fácil avaliar a qualidade da saída.

In [ ]:
from crewai import Task

tarefa_planejamento = Task(
    description=(
        "Planeje um artigo sobre {topico} para {publico}. O objetivo editorial é: {objetivo}. "
        "Analise o contexto de referência fornecido abaixo e use somente informações presentes "
        "nele ou conhecimento geral que não dependa de atualidade. Não invente números, estudos, "
        "citações ou URLs. Marque explicitamente tudo o que precisaria de verificação externa.\n\n"
        "CONTEXTO DE REFERÊNCIA:\n{contexto_referencia}"
    ),
    expected_output=(
        "Um plano editorial abrangente contendo: perfil e necessidades do público; tese; "
        "título provisório; estrutura com introdução, seções e conclusão; pontos essenciais "
        "de cada seção; exemplos possíveis; palavras-chave; fatos fornecidos; ressalvas; e "
        "uma lista separada de afirmações que exigem verificação."
    ),
    agent=planejador,
)

tarefa_redacao = Task(
    description=(
        "Escreva o artigo sobre {topico} seguindo o plano recebido. Use tom {tom} e linguagem "
        "adequada a {publico}. Apoie-se no contexto de referência e preserve as ressalvas do "
        "planejador. Não apresente como fato uma informação marcada para verificação e não "
        "fabrique referências."
    ),
    expected_output=(
        "Um artigo completo em Markdown, entre 500 e 700 palavras, com título, introdução, "
        "de três a cinco seções com subtítulos, exemplos claros e conclusão prática. Não inclua "
        "comentários sobre o processo de escrita."
    ),
    agent=redator,
    context=[tarefa_planejamento],
    markdown=True,
)

tarefa_edicao = Task(
    description=(
        "Edite o artigo produzido para publicação. Verifique alinhamento com o plano, clareza, "
        "progressão lógica, tom {tom}, gramática e formatação Markdown. Remova repetições e "
        "afirmações sem apoio. Não acrescente fatos, estatísticas ou fontes que não apareçam no "
        "contexto. Entregue somente o artigo final."
    ),
    expected_output=(
        "O artigo final em Markdown, pronto para publicação, com título forte, seções coerentes, "
        "conclusão útil e sem notas internas, cercas de código ou prefácios do editor."
    ),
    agent=editor,
    context=[tarefa_planejamento, tarefa_redacao],
    markdown=True,
)

print("Tarefas criadas: planejar, redigir e editar.")

## 5. Reunir agentes e tarefas em uma equipe

A ordem das listas é importante. `Process.sequential` torna explícito que as tarefas serão executadas na sequência planejamento → redação → edição.

In [ ]:
from crewai import Crew, Process

equipe = Crew(
    agents=[planejador, redator, editor],
    tasks=[tarefa_planejamento, tarefa_redacao, tarefa_edicao],
    process=Process.sequential,
    memory=False,
    cache=True,
    max_rpm=1,
    verbose=True,
)

print("Equipe pronta para começar.")

## 6. Iniciar a equipe com `kickoff_async`

Os valores do dicionário substituem `{topico}`, `{publico}`, `{objetivo}`, `{tom}` e `{contexto_referencia}` em todos os agentes e tarefas. Troque-os pelo assunto que desejar. Em notebooks, `kickoff_async` evita conflito com o loop assíncrono já mantido pelo Jupyter.

In [ ]:
entradas = {
    "topico": "Como sistemas multiagentes melhoram fluxos de trabalho",
    "publico": "profissionais de tecnologia que estão começando com agentes de IA",
    "objetivo": "explicar os fundamentos e mostrar quando dividir um problema entre especialistas",
    "tom": "didático, profissional e acessível",
    "contexto_referencia": (
        "Um sistema multiagente reúne agentes especializados. Cada agente possui papel, objetivo "
        "e contexto profissional. No CrewAI, agentes recebem tarefas com descrição e resultado "
        "esperado. Uma Crew reúne agentes e tarefas. As tarefas podem ser sequenciais, paralelas "
        "ou hierárquicas. No fluxo sequencial, a saída de uma etapa alimenta a etapa seguinte. "
        "Papéis e tarefas granulares ajudam cada agente a se concentrar em uma responsabilidade."
    ),
}

resultado = await equipe.kickoff_async(inputs=entradas)

## 7. Exibir o artigo final e as métricas

In [ ]:
from IPython.display import Markdown, display

display(Markdown(resultado.raw))

print("\nMétricas de uso da execução:")
print(equipe.usage_metrics)

## 8. Observar a colaboração

As saídas intermediárias ajudam a entender o que cada especialista acrescentou ao fluxo.

In [ ]:
nomes = ["Planejamento", "Redação", "Edição"]
for nome, saida in zip(nomes, resultado.tasks_output):
    print(f"\n{'=' * 20} {nome} {'=' * 20}\n")
    print(saida.raw[:3000])

## Próximos experimentos

1. Troque o tema, o público e o tom.
2. Forneça um texto de referência maior e compare a precisão.
3. Dê uma ferramenta de pesquisa apenas ao planejador e exija URLs verificáveis.
4. Acrescente um agente de checagem factual entre redator e editor.
5. Compare o resultado da equipe com um único prompt enviado ao mesmo modelo.

Para uso real, não trate o texto gerado como automaticamente verdadeiro: verifique fatos, direitos autorais, dados pessoais e referências antes da publicação.